# Ingeniería de Características (Feature Engineering)

Para este análisis, hemos transformado los datos crudos de precios en métricas financieras avanzadas. Estas variables nos permiten capturar la dinámica del mercado de XRP de una manera que los precios brutos no pueden.

---

### 1. Retornos Logarítmicos (`Log_Returns`)
* **Definición:** Es el logaritmo natural de la razón entre el precio de cierre actual y el precio de cierre anterior: $r_t = \ln(P_t / P_{t-1})$.
* **Utilidad:** En finanzas cuantitativas, se prefieren sobre los retornos simples porque son **aditivos en el tiempo** (puedes sumar retornos diarios para obtener el mensual) y ayudan a normalizar la serie, facilitando el uso de modelos estadísticos.

### 2. Volatilidad Realizada (`Volatility_30d`)
* **Definición:** Es la desviación estándar móvil de los `Log_Returns` calculada sobre una ventana de 30 días.
* **Utilidad:** Mide el **riesgo o incertidumbre** del activo en el corto plazo. Una volatilidad alta indica grandes oscilaciones de precio (miedo o euforia), mientras que una baja indica estabilidad. Es clave para detectar periodos de "pánico" en el mercado.

### 3. Amplitud o Rango Intradía (`Intraday_Range_Pct`)
* **Definición:** Es la diferencia entre el precio máximo (`High`) y el mínimo (`Low`) de un mismo día, normalizada por el precio de apertura: $(High - Low) / Open$.
* **Utilidad:** Mide el **nerviosismo intradía**. A diferencia de los retornos (que solo miran el cierre), esta variable revela qué tan "salvaje" fue la batalla entre compradores y vendedores durante las 24 horas, independientemente de si el precio terminó subiendo o bajando.

---

In [9]:
import pandas as pd
import numpy as np
import os

# 1. Configuración de rutas y carga de datos
file_path = 'items/XRP_Data.csv'
output_path = 'items/XRP_Processed_Data.csv'

df = pd.read_csv(file_path)

# Convertir a datetime y ordenar cronológicamente (paso crítico en series temporales)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')

# --- 2. INGENIERÍA DE VARIABLES (FEATURE ENGINEERING) ---

# A. Retornos Logarítmicos: ln(Precio_t / Precio_t-1)
# Se prefieren en finanzas cuantitativas por su aditividad y propiedades estadísticas.
df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))

# B. Volatilidad Realizada (Rolling Volatility - 30 días):
# Calculamos la desviación estándar móvil de los retornos.
df['Volatility_30d'] = df['Log_Returns'].rolling(window=30).std()

# C. Amplitud o Rango Intradía Normalizado: (High - Low) / Open
# Identifica la magnitud de la fluctuación de precio sin importar la dirección.
df['Intraday_Range_Pct'] = (df['High'] - df['Low']) / df['Open']

# --- 3. LIMPIEZA POST-PROCESAMIENTO ---

# Al usar shift(1) y rolling(30), las primeras filas tendrán valores NaN.
# Las eliminamos para que no afecten las pruebas estadísticas posteriores.
df_clean = df.dropna().copy()

# --- 4. EXPORTACIÓN Y VERIFICACIÓN ---

# Guardar el dataset procesado
df_clean.to_csv(output_path, index=True)

print(f"✅ Proceso completado exitosamente.")
print(f"📊 Registros originales: {len(df)}")
print(f"🧹 Registros tras limpieza: {len(df_clean)}")
print(f"💾 Archivo guardado en: {output_path}")

# Visualización rápida de las nuevas características
display(df_clean[['Close', 'Log_Returns', 'Volatility_30d', 'Intraday_Range_Pct']].head())

✅ Proceso completado exitosamente.
📊 Registros originales: 3044
🧹 Registros tras limpieza: 3014
💾 Archivo guardado en: items/XRP_Processed_Data.csv


,Close,Log_Returns,Volatility_30d,Intraday_Range_Pct
Date,,,,
2017-12-09,0.244708,-0.029859,0.061812,0.061909
2017-12-10,0.237333,-0.030601,0.061268,0.066154
2017-12-11,0.251691,0.058738,0.062015,0.071745
2017-12-12,0.373541,0.394826,0.092994,0.726007
2017-12-13,0.471063,0.231964,0.100643,0.458302


Se realizó un truncamiento inicial del dataset para eliminar el sesgo producido por el periodo de convergencia de los indicadores móviles. Esto garantiza que el análisis estadístico posterior se base exclusivamente en observaciones con métricas de volatilidad y retorno plenamente calculadas, evitando la introducción de ruido mediante imputación de datos

